In [2]:
include("Fun.jl")
include("CRD_STA.jl")
using NonlinearEigenproblems
using Plots
using Interpolations

┌ Warning: attempting to remove probably stale pidfile
│   path = /home/zhj/.julia/compiled/v1.11/DifferentialEquations/UQdwS_Q8qej.ji.pidfile
└ @ FileWatching.Pidfile /home/zhj/.julia/juliaup/julia-1.11.6+0.x64.linux.gnu/share/julia/stdlib/v1.11/FileWatching/src/pidfile.jl:249


In [3]:
const Float = Float64
const ComplexF = ComplexF64
const EigenResult = GeneralizedEigen{ComplexF, ComplexF, Matrix{ComplexF}, Vector{ComplexF}}
struct SimulationParams
    # 输入参数
    N_cheb::Int
    Mr::Float
    gamma::Float
    sigma::Float
    Ro::Float
    Tw::Float
    
    # 计算参数
    Co::Float
    
    # 基础流变量
    u0::Vector{Float}
    v0::Vector{Float}
    w0::Vector{Float}
    f::Vector{Float}
    q::Vector{Float}
    D::Matrix{Float}
    D2::Matrix{Float}
    x::Matrix{Float}
    
    # 核心矩阵
    F::Matrix{Float}
    G::Matrix{Float}
    H::Matrix{Float}
    T::Matrix{Float}
    
    # 物理场向量
    rho::Matrix{Float}
    lam::Matrix{Float}
    kappa::Matrix{Float}
   function SimulationParams(N_cheb::Int, Mr::Float, gamma::Float, sigma::Float, Ro::Float, Tw::Float)
        # 计算不变值
        Co = 2 - Ro - Ro^2
        
        # 计算基础流 (假设这些函数已定义)
        u0, v0, w0, f, q, D, D2, x = baseflow_var(N_cheb, Ro, "cheb")
        
        # 计算温度和焓场 (假设T_ca函数已定义)
        H_mat, T_mat = T_ca(Mr, f, q, w0, gamma, Tw)
        
        # 插值计算 (假设interp函数已定义)
        F_mat, G_mat, H_mat, T_mat, rho, z = interp(u0, v0, H_mat, T_mat, x, N_cheb, "sim")
        
        # 计算粘性系数
        lam = - (2/3) .* T_mat
        kappa = (1/sigma) .* T_mat
        
        # 创建并返回结构体
        new(N_cheb, Mr, gamma, sigma, Ro, Tw, Co, 
            u0, v0, w0, f, q, D, D2, x,
            F_mat, G_mat, H_mat, T_mat,
            rho, lam, kappa)
        end
end
function setup_simulation(N_cheb, Mr, gamma, sigma, Ro, Tw)
    return SimulationParams(N_cheb, Mr, gamma, sigma, Ro, Tw)
end
function BaseFlow(N_cheb,Mr,gamma,sigma,Ro,Tw)
    params = setup_simulation(N_cheb,Mr,gamma,sigma,Ro,Tw)
    const_params = (
    params.F, params.G, params.H, params.rho, params.lam, params.kappa, params.D,
    params.D2,params.T, params.sigma, params.gamma, params.N_cheb, params.Ro, params.Co
    )
    return params,const_params
end


BaseFlow (generic function with 1 method)

In [4]:
function eigenpro(R, Ma, al, be, const_params)
    F, G, H, rho, lam, kappa, D, D2,T, sigma, gamma, N_cheb, Ro, Co = const_params
    B0, B1 = Timemode(F, G, H, rho, lam, kappa, T, sigma, gamma, R, Ma, al, be, N_cheb, Ro, Co,D ,D2)
    return B0,B1
end
function eigensolve(B0,B1)
    return eigen(B0,B1)
end
function selectdval(C,re_max)
    return filter(x->-0.05<imag(x)<0.05&&abs(real(x))<re_max,C.values)
end
function pinpoint(R,be,total,alpha,step)
    pinpoints = Matrix{Complex}(undef,0,4)

    for i = 1 : size(total,2)  

        d2 = diff1(real(total[:,i]),step)

        for j = 1 : length(d2)-1

            if d2[j] * d2[j+1] < 0 && abs(d2[j+1])<0.01
                pinpoints = [pinpoints;[R alpha[j+1] be total[j+1,i]]]

            end

        end
    end
    return pinpoints
end
function ali_range(R,ali,re_max,Mr,be,const_params,num)
    total = []
    Ma = Mr/R
    al_all =  Matrix{Complex}(undef,0,1)
    alr_ref = 0.35
    al = complex(alr_ref,ali)
    B0,B1 = eigenpro(R, Ma, al, be, const_params)
    C = eigensolve(B0,B1)
    val = selectdval(C,re_max)
        for i = 1 : min(num,length(val))
            indi1 = []
            val_temp = val[i]
            vec = eigvector(val[i],C.values,C.vectors)

            for alr in range(alr_ref,0.6,step = 0.002)
                al = complex(alr,ali)
                B0,B1 = eigenpro(R, Ma, al, be, const_params)
                val0,vec0 = RQI(B0,B1,val_temp,q0 = vec)
                indi1 = [indi1;val0]
                al_all = [al_all;al]
                val_temp = val0
                vec = vec0 
            end
            indi2 = []
            val_temp = val[i]
            vec = eigvector(val[i],C.values,C.vectors)
            for alr in range(alr_ref,0.25,step = -0.002)
                al = complex(alr,ali)
                B0,B1 = eigenpro(R, Ma, al, be, const_params)
                val0,vec0 = RQI(B0,B1,val_temp,q0 = vec)
                indi2 = [indi2;val0]
                al_all = [al_all;al]
                val_temp = val0
                vec = vec0 
            end
            indi = [indi2[end:-1:1,:];indi1]
            if i == 1
                total = indi
            else
                total = [total indi]
            end
        end
    return total,al_all
end
function acceleration_ali(data_all)
    if isreal(data_all[end,3]) 
        step = 0.01
    else 
        step = 0.002
    end
    step = 0.002
    return step
end
function acceleration_R(data_all,Ro)
    if Ro == 1
        if data_all[end,6] < -0.0015
            step = 2
        elseif -0.0015 < data_all[end,6] < -0.0003
            step = 0.5
        else
            step = 0.15
        end
    else
        if data_all[end,1] > 100
            step = 2
        elseif data_all[end,6] < -0.0015
            step = 2
        elseif -0.0015 < data_all[end,6] < -0.0003
            step = 1
        else
            step = 0.2
        end
    end
    return step
end

acceleration_R (generic function with 1 method)

In [14]:
function caculate(beta,const_params, Tw,Mr,Ro)
    dataall = Matrix{Float}(undef,0,6)
    re_max = 0.2
    data_all = Matrix{Complex}(undef,0,4)
    backstep = 2

    # if Ro == 1
    #     ali_ref = 0
    # else
    #     ali_ref = 0.26 - (Tw - 1) * 0.16
    # end
    ali_ref = 0.13
    for be in beta

        while true
            data_all = Matrix{Complex}(undef,0,4)
            data =  Matrix{Complex}(undef,0,4)
            data_1 = data_all = Matrix{Complex}(undef,0,4)
            if Ro == 1
                if size(dataall,1) != 0
                    R = max(dataall[end,1] - backstep , 20)
                else
                    R = 22
                end
            else
                R = 180
                val = [0]
                while imag(val[1])> - 0.002
                    R -= 5
                    Ma = Mr/R
                    al = complex(0.35,ali_ref)
                    B0,B1 = eigenpro(R, Ma, al, be, const_params)
                    val = selectdval(eigensolve(B0,B1),re_max)
                    writedlm("AS.dat",val[1])

                end
            end

            while true 
                total,al_all = ali_range(R,ali_ref,re_max,Mr,be,const_params,2)
                PinPoint = pinpoint(R,be,total,al_all,0.002)
                if size(PinPoint,1) == 0
                    ali_ref += 0.002
                    writedlm("AS.dat",ali_ref)
                    continue

                else
                    data_all = [data_all;[R be PinPoint[findmax(imag(PinPoint[:,4]))[2],2] PinPoint[findmax(imag(PinPoint[:,4]))[2],4]]]
                    data_1 = [data_1;[R be PinPoint[findmax(imag(PinPoint[:,4]))[2],2] PinPoint[findmax(imag(PinPoint[:,4]))[2],4]]]
                end

                writedlm("AS.dat",data_all)
                
                if length(axes(data_1,1)) > 2 && imag(data_1[end - 1,4]) > imag(data_1[end-2,4]) && imag(data_1[end-1,4]) > imag(data_1[end,4]) && imag(data_1[end - 1,4]) != 0 && abs(real(data_1[end - 1 ,4])-real(data_1[end,4])) < 0.005 && abs(real(data_1[end - 1 ,4])-real(data_1[end - 2,4])) < 0.005
                    break
                end
                step_ali = acceleration_ali(data_all)
                ali_ref += step_ali

            end
            #section2
            ali = imag(data_all[end - 1,3]) 
            R0 = real(data_all[end - 1,1])
            fill!(data_all,0)
            data1 =  Matrix{Complex}(undef,0,4)
            while true
                Ma = Mr/R
                total,al_all = ali_range(R,ali,re_max,Mr,be,const_params,2)
                PinPoint = pinpoint(R,be,total,al_all,0.002)

                if size(PinPoint,1) == 0
                    R += 0.15
                    continue
                else
                    data_temp = [R be PinPoint[findmax(imag(PinPoint[:,4]))[2],2] PinPoint[findmax(imag(PinPoint[:,4]))[2],4]]
                end
                data1 = [data1;data_temp]
                data_all = [real(data1[:,1]) real(data1[:,2]) real(data1[:,3]) imag(data1[:,3]) real(data1[:,4]) imag(data1[:,4])]
                writedlm("AS1.dat",data_all)
                if (data_all[end,6]>-1e-4)
                    dataall = [dataall;data_all[end:end,:]]
                end
                if (data_all[end,6]>-1e-4) 
                    # || (size(data_all,1) > 2 && (data_all[end,6] < data_all[end-1,6]) && (data_all[end-1,6]>data_all[end-2,6]))  
                   break
                end 
                step_R = acceleration_R(data_all,Ro)
                if be < -0.35
                    R += 5*step_R
                else
                    R += step_R
                end
            end
            if (data_all[end,6]>-1e-4) 
                # re_max = abs(real(dataall[end,5])) + 0.03
                ali_ref = real(dataall[end,4]) - 0.03
                writedlm("Tw=$(Tw)_Ma=$(Mr)_Ro=$(Ro).dat",dataall)
                backstep = 1
                break
            else 
                ali_ref -= 0.03
                backstep -= 0.4
            end

        end
    end
end    

caculate (generic function with 1 method)

In [15]:
N_cheb = 59
gamma = 1.4
sigma = 0.72
Ro = 0.687
Tw = 1.0
Mr = 0.3
# R = 250
# ali_ref = 0.18
# be = -0.01
# re_max = 0.2
# Ma = Mr/R
# params,const_params = BaseFlow(N_cheb, Mr, gamma, sigma, Ro, Tw)
# al = complex(0.35,ali_ref)
# B0,B1 = eigenpro(R, Ma, al, be, const_params)
# val = selectdval(eigensolve(B0,B1),re_max)
# total,al_all = ali_range(R,ali_ref,re_max,Mr,be,const_params,2)
# PinPoint = pinpoint(R,be,total,al_all,0.002)
for Mr = 0.6
    params,const_params = BaseFlow(N_cheb, Mr, gamma, sigma, Ro, Tw) 
    beta = range(0.14,0.4,step = 0.01) 
    caculate(beta,const_params,Tw,Mr,Ro) 
end

In [7]:
scatter(real.(val),imag.(val))

UndefVarError: UndefVarError: `val` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [10]:
Ro = 0.687
omega=-0.5
for Mr = 0.3
    for Tw = 0.8
        str1 = "Zone T= \"Ma=$Mr,Tw=$Tw,Ro=$Ro,INTER\" "
        str2 = "Variables=\"beta\" \"R\" "
        str3 = "Variables=\"r\" \"beta\" \"alpha_i\" \"alpha_r\" \"omega_i\" \"omega_r\" "
        str4 = "Zone T= \"Ma=$Mr,Tw=$Tw,Ro=$Ro,ORIG\" "

        a1 = readdlm("omega=$(omega)Tw=$(Tw),Ma=$(Mr).dat")
        k = 2      
        p = sortperm(a1[:, k])
        a1 = a1[p, :]  
        x = a1[:,2]
        y = a1[:,1] 
        function sort_unique_xy(x::AbstractVector, y::AbstractVector)
            @assert length(x) == length(y) "x,y 长度不一致"
            p = sortperm(x)
            xs, ys = x[p], y[p]
            # 去掉重复 x（保留第一次出现）
            keep = [true; diff(xs) .> 0]
            return xs[keep], ys[keep]
        end
        xs, ys = sort_unique_xy(x, y)
        xq = range(min(xs[1],0), max(xs[end],0.41), length=500)
        yq_quad = pchip_interp_extrap(xs, ys, collect(xq); m=5, mode=:quad)
        function smooth_moving_avg(y; w)
            @assert isodd(w) "窗口 w 请设为奇数"
            r = (w-1)÷2
            ys = similar(y, Float64)
            for i in eachindex(y)
                lo = max(firstindex(y), i-r)
                hi = min(lastindex(y),  i+r)
                ys[i] = Statistics.mean(@view y[lo:hi])
            end
            return ys
        end
        y_ma = smooth_moving_avg(yq_quad; w= eleven=7)  # 写法示例：w=11
        all = [xq y_ma]  
        open("Tw=$(Tw)_Ma=$(Mr)_Ro=$(Ro)_interp.dat","w") do f
            println(f, str2)
            println(f, str1)
            writedlm(f, all)
        end
        open("Tw=$(Tw)_Ma=$(Mr)_Ro=$(Ro)_orig.dat","w") do f
            println(f, str3)
            println(f, str4)
            writedlm(f, a1)
        end
    end
end

ArgumentError: ArgumentError: Cannot open 'omega=-0.5Tw=0.8,Ma=0.3.dat': not a file

In [11]:
plot(yq_quad,xq)
plot!(y_ma,xq)

UndefVarError: UndefVarError: `yq_quad` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
# ——— 高精度一维插值 + 外推（已修正，无“2h”“3s[1]”之类语法错误） ———
# 依赖：仅 Base/LinearAlgebra
using LinearAlgebra

# 计算 PCHIP 端点与内点斜率（Fritsch–Carlson + Fritsch–Butland 端点处理）
function _pchip_slopes(x::AbstractVector{<:Real}, y::AbstractVector{<:Real})
    n = length(x)
    @assert n == length(y) ≥ 2 "x,y 长度需一致且≥2"
    @assert issorted(x) "x 必须升序"

    h = @views x[2:end] .- x[1:end-1]                 # 段长
    s = @views (y[2:end] .- y[1:end-1]) ./ h          # 分段斜率 Δy/Δx

    d = Vector{Float64}(undef, n)

    # n == 2 时只有一段：两端导数都等于该段斜率
    if n == 2
        d[1] = s[1]; d[2] = s[1]
        return d
    end

    # 端点导数（Fritsch–Butland 建议），然后做单调性限制
    d[1] = ((2*h[1] + h[2]) * s[1] - h[1] * s[2]) / (h[1] + h[2])
    if sign(d[1]) != sign(s[1])
        d[1] = 0.0
    elseif sign(s[1]) != sign(s[2]) && abs(d[1]) > 3 * abs(s[1])
        d[1] = 3 * s[1]
    end

    d[n] = ((2*h[end] + h[end-1]) * s[end] - h[end] * s[end-1]) / (h[end] + h[end-1])
    if sign(d[n]) != sign(s[end])
        d[n] = 0.0
    elseif sign(s[end]) != sign(s[end-1]) && abs(d[n]) > 3 * abs(s[end])
        d[n] = 3 * s[end]
    end

    # 内点导数（Fritsch–Carlson 加权调和平均，保形避免振荡）
    @inbounds for i in 2:n-1
        if s[i-1] == 0 || s[i] == 0 || sign(s[i-1]) != sign(s[i])
            d[i] = 0.0
        else
            w1 = 2*h[i] + h[i-1]
            w2 = h[i] + 2*h[i-1]
            d[i] = (w1 + w2) / (w1 / s[i-1] + w2 / s[i])
        end
    end

    return d
end

# Hermite 段值
@inline function _hermite(x0, x1, y0, y1, m0, m1, x)
    t  = (x - x0) / (x1 - x0)
    t2 = t * t
    t3 = t2 * t
    h00 =  2*t3 - 3*t2 + 1
    h10 =      t3 - 2*t2 + t
    h01 = -2*t3 + 3*t2
    h11 =      t3 -   t2
    return h00*y0 + h10*(x1 - x0)*m0 + h01*y1 + h11*(x1 - x0)*m1
end

# 端点附近 m 个点的二次最小二乘外推（m≥3 自动夹取）
function _quad_extrap_edge(x::AbstractVector{<:Real}, y::AbstractVector{<:Real},
                           xq::Real; left::Bool, m::Int=4)
    n  = length(x)
    @assert n == length(y) ≥ 3 "二次外推建议≥3点"
    mm = max(3, min(m, n))                         # 至少3点
    xx = left ? @view(x[1:mm]) : @view(x[end-mm+1:end])
    yy = left ? @view(y[1:mm]) : @view(y[end-mm+1:end])

    # Vandermonde 最小二乘：y ≈ a + b x + c x^2
    A = ones(length(xx), 3)
    @views A[:, 2] .= xx
    @views A[:, 3] .= xx .^ 2
    a, b, c = A \ yy
    return a + b * xq + c * xq^2
end

"""
    pchip_interp_extrap(xs, ys, xq; m=4, mode=:quad)

区间内：PCHIP（三次 Hermite，形状保持）。
区间外：
- mode = :quad → 端点附近 m 点二次拟合外推（精度更高）
- mode = :line → 用 PCHIP 端点导数线性外推（更稳更保形）

支持标量或向量 xq。
"""
function pchip_interp_extrap(xs::AbstractVector{<:Real},
                             ys::AbstractVector{<:Real},
                             xq::Union{Real,AbstractVector{<:Real}}; m::Int=4, mode::Symbol=:quad)
    @assert issorted(xs) "xs 必须升序"
    @assert length(xs) == length(ys) ≥ 2

    d = _pchip_slopes(xs, ys)

    # 标量查询
    if xq isa Real
        if xq ≤ xs[1]
            return (mode === :line) ? (ys[1] + d[1] * (xq - xs[1])) :
                                      _quad_extrap_edge(xs, ys, xq; left=true, m=m)
        elseif xq ≥ xs[end]
            return (mode === :line) ? (ys[end] + d[end] * (xq - xs[end])) :
                                      _quad_extrap_edge(xs, ys, xq; left=false, m=m)
        else
            i = searchsortedlast(xs, xq)   # xs[i] ≤ xq < xs[i+1]
            return _hermite(xs[i], xs[i+1], ys[i], ys[i+1], d[i], d[i+1], xq)
        end
    end

    # 向量查询
    xqv = xq
    yq  = similar(xqv, Float64)
    @inbounds for j in eachindex(xqv)
        xqi = xqv[j]
        if xqi ≤ xs[1]
            yq[j] = (mode === :line) ? (ys[1] + d[1] * (xqi - xs[1])) :
                                       _quad_extrap_edge(xs, ys, xqi; left=true, m=m)
        elseif xqi ≥ xs[end]
            yq[j] = (mode === :line) ? (ys[end] + d[end] * (xqi - xs[end])) :
                                       _quad_extrap_edge(xs, ys, xqi; left=false, m=m)
        else
            i = searchsortedlast(xs, xqi)
            yq[j] = _hermite(xs[i], xs[i+1], ys[i], ys[i+1], d[i], d[i+1], xqi)
        end
    end
    return yq
end

# ——— 用法示例 ———
# 假设你已有升序去重后的数据 (xs, ys)
ΔL = xs[2] - xs[1]; ΔR = xs[end] - xs[end-1]
xq = range(xs[1] - 0*ΔL, xs[end] + 8*ΔR, length=400)
yq_quad = pchip_interp_extrap(xs, ys, collect(xq); m=5, mode=:quad)
# yq_line = pchip_interp_extrap(xs, ys, collect(xq); mode=:line)


UndefVarError: UndefVarError: `xs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:
# plot(y , x)
